In [1]:
# TÍTULO: Entrenamiento del Modelo OCR (Clasificación de Caracteres)
# OBJETIVO: Entrenar una red YOLOv8-Classifier para reconocer letras y números individuales.

import torch
from ultralytics import YOLO
import os
import shutil
import glob
import matplotlib.pyplot as plt
import cv2
from datetime import datetime

# --- VERIFICACIÓN DE GPU ---
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Activa: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    print("ALERTA: Usando CPU. El entrenamiento será más lento.")
    device = 'cpu'

# --- RUTAS ---
# Nota: Apuntamos a la carpeta donde el Notebook 01 guardó los recortes organizados
# Estructura esperada: ../../datasets/03_caracteres/{train,val}/{A,B,C...}
DATASET_DIR = '../../datasets/03_caracteres' 

# Salida de modelos
PROJECT_DIR = '../../models/03_caracteres'

# Verificar que el dataset existe
if os.path.exists(DATASET_DIR):
    print(f"Dataset encontrado en: {os.path.abspath(DATASET_DIR)}")
    # Contar clases
    train_dir = os.path.join(DATASET_DIR, 'train')
    if os.path.exists(train_dir):
        classes = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
        print(f"Clases detectadas: {len(classes)} ({sorted(classes)[:5]}...)")
else:
    print(f"ERROR: No se encuentra el dataset en {DATASET_DIR}")

PyTorch Version: 2.9.1+cu126
GPU Activa: NVIDIA GeForce RTX 4070 Ti
Dataset encontrado en: /home/roberto/moca_proyecto/datasets/03_caracteres
Clases detectadas: 36 (['0', '1', '2', '3', '4']...)


In [3]:
# Cargar modelo de Clasificación (Nano)
# Usamos la versión '-cls'
model_variant = 'yolov8n-cls.pt'
model = YOLO(model_variant)

# Nombre del experimento
run_name = f"{datetime.now().strftime('%Y%m%d')}_ocr_v8n_cls"

print(f"Modelo cargado: {model_variant}")
print(f"Carpeta de salida: {PROJECT_DIR}/{run_name}")

Modelo cargado: yolov8n-cls.pt
Carpeta de salida: ../../models/03_caracteres/20251203_ocr_v8n_cls


In [3]:
# --- INICIAR ENTRENAMIENTO ---
# YOLO Classification no necesita un archivo .yaml, solo la ruta de la carpeta raíz.
results = model.train(
    data=DATASET_DIR,   # Ruta a la carpeta que contiene train/ y val/
    project=PROJECT_DIR,
    name=run_name,
    
    # Hiperparámetros
    epochs=20,          # 10-20 épocas suelen bastar para OCR sintético limpio
    patience=5,         
    batch=64,           # Lotes más grandes porque las imágenes son pequeñas (64px)
    imgsz=64,           # Resolución de entrada (caracteres son pequeños)
    
    # Optimizaciones
    pretrained=True,
    optimizer='auto',
    
    # Aumentación (Leve, para no deformar las letras)
    degrees=5.0,        # Rotación leve
    fliplr=0.0,         # NO espejear (una 'E' al revés no sirve)
    
    device=device,
    verbose=True,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/03_caracteres, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251202_ocr_v8n_cl

In [4]:
print("Entrenamiento finalizado. Validando...")

# Cargar el mejor modelo resultante
best_model_path = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Ejecutar validación
metrics = best_model.val()

print("\nRESULTADOS OCR:")
# top1: Qué tan seguido le atina a la primera
print(f"   Top-1 Accuracy: {metrics.top1:.4f}")
# top5: Qué tan seguido la respuesta correcta está en sus 5 mejores opciones
print(f"   Top-5 Accuracy: {metrics.top5:.4f}")

Entrenamiento finalizado. Validando...
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,480,996 parameters, 0 gradients, 3.3 GFLOPs
train: /home/roberto/moca_proyecto/datasets/03_caracteres/train... found 5504 images in 36 classes ✅ 
val: /home/roberto/moca_proyecto/datasets/03_caracteres/val... found 1373 images in 36 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 124.7±42.8 MB/s, size: 1.8 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/03_caracteres/val... 1373 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1373/1373 4.2Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 86/86 269.0it/s 0.3s0.3s
                   all      0.792      0.959
Speed: 0.0ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/roberto/moca_proyecto/notebooks/03_caracteres/runs/classify/val

RESULTADOS OCR:
   Top-1 Accuracy: 0.

In [6]:
# Cargar modelo de Clasificación (Nano) por 100 épocas
# Usamos la versión '-cls'
model_variant = 'yolov8n-cls.pt'
model = YOLO(model_variant)

# Nombre del experimento
run_name = f"{datetime.now().strftime('%Y%m%d')}_ocr_v8n_cls_100ep"

print(f"Modelo cargado: {model_variant}")
print(f"Carpeta de salida: {PROJECT_DIR}/{run_name}")

Modelo cargado: yolov8n-cls.pt
Carpeta de salida: ../../models/03_caracteres/20251202_ocr_v8n_cls_100ep


In [7]:
# --- INICIAR ENTRENAMIENTO de 100 ÉPOCAS ---
# YOLO Classification no necesita un archivo .yaml, solo la ruta de la carpeta raíz.
results = model.train(
    data=DATASET_DIR,   # Ruta a la carpeta que contiene train/ y val/
    project=PROJECT_DIR,
    name=run_name,
    
    # Hiperparámetros
    epochs=100,          # 10-20 épocas suelen bastar para OCR sintético limpio
    patience=5,         
    batch=64,           # Lotes más grandes porque las imágenes son pequeñas (64px)
    imgsz=64,           # Resolución de entrada (caracteres son pequeños)
    
    # Optimizaciones
    pretrained=True,
    optimizer='auto',
    
    # Aumentación (Leve, para no deformar las letras)
    degrees=5.0,        # Rotación leve
    fliplr=0.0,         # NO espejear (una 'E' al revés no sirve)
    
    device=device,
    verbose=True,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/03_caracteres, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251202_ocr_v8n_c

In [8]:
print("Entrenamiento finalizado. Validando...")

# Cargar el mejor modelo resultante
best_model_path = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Ejecutar validación
metrics = best_model.val()

print("\nRESULTADOS OCR:")
# top1: Qué tan seguido le atina a la primera
print(f"   Top-1 Accuracy: {metrics.top1:.4f}")
# top5: Qué tan seguido la respuesta correcta está en sus 5 mejores opciones
print(f"   Top-5 Accuracy: {metrics.top5:.4f}")

Entrenamiento finalizado. Validando...
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,480,996 parameters, 0 gradients, 3.3 GFLOPs
train: /home/roberto/moca_proyecto/datasets/03_caracteres/train... found 5504 images in 36 classes ✅ 
val: /home/roberto/moca_proyecto/datasets/03_caracteres/val... found 1373 images in 36 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 140.2±48.1 MB/s, size: 1.8 KB)
val: Scanning /home/roberto/moca_proyecto/datasets/03_caracteres/val... 1373 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1373/1373 5.0Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 86/86 312.0it/s 0.3s0.2s
                   all       0.86      0.968
Speed: 0.0ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/roberto/moca_proyecto/notebooks/03_caracteres/runs/classify/val2

RESULTADOS OCR:
   Top-1 Accuracy: 0

In [9]:
import random

# Buscar una imagen aleatoria del set de validación
# La estructura es val/CLASE/imagen.jpg
val_dir = os.path.join(DATASET_DIR, 'val')
if os.path.exists(val_dir):
    # Obtener todas las clases
    classes = os.listdir(val_dir)
    if classes:
        # Elegir clase y foto al azar
        rand_class = random.choice(classes)
        class_path = os.path.join(val_dir, rand_class)
        images = glob.glob(os.path.join(class_path, '*.jpg'))
        
        if images:
            test_img = random.choice(images)
            
            # Predecir
            # probs=True devuelve las probabilidades
            results = best_model(test_img)
            
            # Obtener la clase predicha
            pred_class_id = results[0].probs.top1
            pred_class_name = results[0].names[pred_class_id]
            confidence = results[0].probs.top1conf.item()

            # Mostrar
            img_array = cv2.imread(test_img)
            plt.figure(figsize=(4,4))
            plt.imshow(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
            plt.title(f"Real: {rand_class} | Pred: {pred_class_name} ({confidence:.2%})")
            plt.axis('off')
            plt.show()
        else:
            print("No hay imágenes en la clase seleccionada.")
    else:
        print("No hay carpetas de clases en val.")
else:
    print("No existe el directorio de validación.")


image 1/1 /home/roberto/moca_proyecto/notebooks/03_caracteres/../../datasets/03_caracteres/val/9/VNS-769_8842_5.jpg: 64x64 9 1.00, Q 0.00, Y 0.00, S 0.00, 0 0.00, 1.7ms
Speed: 0.7ms preprocess, 1.7ms inference, 0.0ms postprocess per image at shape (1, 3, 64, 64)


<Figure size 400x400 with 1 Axes>

In [10]:
PRODUCTION_PATH = '../../production_weights'
os.makedirs(PRODUCTION_PATH, exist_ok=True)

target_path = os.path.join(PRODUCTION_PATH, 'ocr_best.pt')
shutil.copy(best_model_path, target_path)

print(f"Modelo OCR exportado a: {target_path}")

Modelo OCR exportado a: ../../production_weights/ocr_best.pt


In [5]:
# Cargar modelo de Clasificación (Nano) para 30000 imágenes
# Usamos la versión '-cls'
model_variant = 'yolov8n-cls.pt'
model_30k = YOLO(model_variant)

# Nombre del experimento
run_name = f"{datetime.now().strftime('%Y%m%d')}_ocr_v8n_cls_30k"

print(f"Modelo cargado: {model_variant}")
print(f"Carpeta de salida: {PROJECT_DIR}/{run_name}")

Modelo cargado: yolov8n-cls.pt
Carpeta de salida: ../../models/03_caracteres/20251203_ocr_v8n_cls_30k


In [6]:
# --- INICIAR ENTRENAMIENTO ---
# YOLO Classification no necesita un archivo .yaml, solo la ruta de la carpeta raíz.
results = model_30k.train(
    data=DATASET_DIR,   # Ruta a la carpeta que contiene train/ y val/
    project=PROJECT_DIR,
    name=run_name,
    
    # Hiperparámetros
    epochs=20,          # 10-20 épocas suelen bastar para OCR sintético limpio
    patience=5,         
    batch=64,           # Lotes más grandes porque las imágenes son pequeñas (64px)
    imgsz=64,           # Resolución de entrada (caracteres son pequeños)
    
    # Optimizaciones
    pretrained=True,
    optimizer='auto',
    
    # Aumentación (Leve, para no deformar las letras)
    degrees=5.0,        # Rotación leve
    fliplr=0.0,         # NO espejear (una 'E' al revés no sirve)
    
    device=device,
    verbose=True,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.233 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11852MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/03_caracteres, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251203_ocr_v8n_cl

KeyboardInterrupt: 

In [ ]:
print("Entrenamiento finalizado. Validando...")

# Cargar el mejor modelo resultante
best_model_path = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Ejecutar validación
metrics = best_model.val()

print("\nRESULTADOS OCR:")
# top1: Qué tan seguido le atina a la primera
print(f"   Top-1 Accuracy: {metrics.top1:.4f}")
# top5: Qué tan seguido la respuesta correcta está en sus 5 mejores opciones
print(f"   Top-5 Accuracy: {metrics.top5:.4f}")

In [ ]:
import random

# Buscar una imagen aleatoria del set de validación
# La estructura es val/CLASE/imagen.jpg
val_dir = os.path.join(DATASET_DIR, 'val')
if os.path.exists(val_dir):
    # Obtener todas las clases
    classes = os.listdir(val_dir)
    if classes:
        # Elegir clase y foto al azar
        rand_class = random.choice(classes)
        class_path = os.path.join(val_dir, rand_class)
        images = glob.glob(os.path.join(class_path, '*.jpg'))
        
        if images:
            test_img = random.choice(images)
            
            # Predecir
            # probs=True devuelve las probabilidades
            results = best_model(test_img)
            
            # Obtener la clase predicha
            pred_class_id = results[0].probs.top1
            pred_class_name = results[0].names[pred_class_id]
            confidence = results[0].probs.top1conf.item()

            # Mostrar
            img_array = cv2.imread(test_img)
            plt.figure(figsize=(4,4))
            plt.imshow(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
            plt.title(f"Real: {rand_class} | Pred: {pred_class_name} ({confidence:.2%})")
            plt.axis('off')
            plt.show()
        else:
            print("No hay imágenes en la clase seleccionada.")
    else:
        print("No hay carpetas de clases en val.")
else:
    print("No existe el directorio de validación.")